In [1]:
# jax ecosystem
import jax

# jax.config.update("jax_enable_x64", False)
jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_platform_name", "gpu")
jax.config.update("jax_debug_nans", False)

device = jax.local_devices()[0]
print(device.device_kind)

import jax.numpy as np
import jax.tree as jtu
import jax.random as jr

# amigo
import amigo
import dLux
import zodiax

# matplotlib ecosystem
import matplotlib.pyplot as plt
import matplotlib as mpl
import ehtplot
import scienceplots

# other
import pandas as pd
import os
import sys

# matplotlib parameters
plt.style.use(["science", "bright", "no-latex"])
plt.rcParams["image.cmap"] = "inferno"
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = "lower"
plt.rcParams["figure.dpi"] = 300
plt.rcParams["font.size"] = 8
# plt.rcParams["xtick.direction"] = "out"
plt.rcParams["ytick.direction"] = "out"

def get_cmap(cmap_name: str):
    cmap = mpl.colormaps[cmap_name]
    cmap.set_bad("k", 0.5)
    return cmap

inferno = get_cmap("inferno")
seismic = get_cmap("seismic")
coolwarm = get_cmap("coolwarm")

load_dict = lambda x: np.load(f"{x}", allow_pickle=True).item()  # helper function

if jax.config.read("jax_enable_x64"):
    print("64bit enabled")
else:
    print("32bit enabled")

import equinox
import lineax

print("jax version:", jax.__version__)
print("equinox version:", equinox.__version__)
print("lineax version", lineax.__version__)
print("zodiax version:", zodiax.__version__)
print("dLux version:", dLux.__version__)
print("amigo version:", amigo.__version__)

ERROR:2026-08-19 05:24:05,049:jax._src.xla_bridge:475: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/dgxuser/max/conda/envs/oldamigo/lib/python3.12/site-packages/jax/_src/xla_bridge.py", line 473, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/dgxuser/max/conda/envs/oldamigo/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/dgxuser/max/conda/envs/oldamigo/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load the CUDA libraries.


cpu
64bit enabled
jax version: 0.8.3
equinox version: 0.13.7
lineax version 0.1.0
zodiax version: 0.4.1
dLux version: 0.14.3
amigo version: 0.0.10


# Loading in data

In [2]:
# Setting data path
from socket import gethostname
from retrain_fns import Tee

print(f"host name: {gethostname()}")

# compile caching
if device.device_kind != "cpu":
    if gethostname().startswith("max-"):
        jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
    else:
        jax.config.update("jax_compilation_cache_dir", "/fred/oz440/max/jax_cache")

    jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
    jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
    jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")

    stats = device.memory_stats()
    free = stats["bytes_limit"] - stats["bytes_in_use"]
    print(f"Total VRAM : {stats['bytes_limit'] / 1024**3:.2f} GB")
    print(f"In use     : {stats['bytes_in_use'] / 1024**3:.2f} GB")
    print(f"Free       : {free / 1024**3:.2f} GB")


if (
    gethostname() == "maxs-mbp-14.shared.sydney.edu.au"
    or gethostname() == "Maxs-MacBook-Pro-14.local"
):
    data_dir = "/Volumes/research-data/PRJ-PAT/max/data/JWST/"
    cache_dir = "/Volumes/research-data/PRJ-PAT/max/data/amigo_cache"
    amigo_files_path = "/Volumes/research-data/PRJ-PAT/max/data/amigo_files/v_0.0.10"
    output_path = "/Users/mc/nt_outputs/local_retrain"

elif gethostname().startswith("max-"):
    data_dir = "/home/dgxuser/max/data/JWST/"
    cache_dir = "/home/dgxuser/max/data/amigo_cache"
    amigo_files_path = "/home/dgxuser/max/data/amigo_files/v_0.0.10"
    output_path = "/home/dgxuser/max/outputs/retrain"

else:
    data_dir = "/fred/oz440/max/data/JWST/"
    cache_dir = "/fred/oz440/max/data/amigo_cache"
    amigo_files_path = "/fred/oz440/max/data/amigo_files/v_0.0.10"
    output_path = "/fred/oz440/max/outputs/retrain"

# cleaning directory of empty folders
for d in os.listdir(output_path):
    this_dir = os.path.join(output_path, d)
    fs = os.listdir(this_dir)
    if len(fs) == 0:
        os.rmdir(this_dir)

# dealing with saving figures when running in script
def check_script():
    if "__file__" in globals():
        print("Running as a script")
        return True
    else:
        print("Running in a notebook or interactive shell")
        return False

# Quick toggle flags
epochs = 15
n_files_per_filt = 2
one_filt_flag = True

print(f"amigo_files_path: {amigo_files_path}")

host name: max-cpuv3-0-6
amigo_files_path: /home/dgxuser/max/data/amigo_files/v_0.0.10


In [3]:
from retrain_fns import summarise_files
import importlib

resource_path = importlib.resources.files(amigo).joinpath("data/badpix.npy")

with importlib.resources.as_file(resource_path) as file_path:
    badpix = np.array(np.load(file_path), dtype=int)

def add_badpix(file):
    file["BADPIX"].data = badpix
    if file[0].header["EXP_TYPE"] == "NIS_DARK":
        file["BADPIX"].data[25, 74] = 1
    # if file[0].header["PROGRAM"] == "04481" and file[0].header["FILTER"] == "F430M":
    #     file["BADPIX"].data[30, 40] = 1
    return file


## Validation files
Calibrators from other science programs

In [4]:
def sort_files(files, n):
    files.sort(key=lambda x: (x[0].header["PROGRAM"], x[0].header["FILTER"], x[0].header["PATT_NUM"]), reverse=True)
    filts = ["F480M", "F430M", "F380M"]
    these_files = {}
    for filt in filts:
        x = []
        for file in files:
            if file[0].header["FILTER"] == filt:
                if one_filt_flag and filt != "F480M":
                    continue
                x.append(file)
        these_files[filt] = x[0:n]
    files = [f for sub_f in these_files.values() for f in sub_f]
    return files

## Calibration files
The main training data, including flats.

In [5]:
# LOADING IN DATA
from astropy.io import fits
from amigo.files import get_files

calbin_files = []

print()
print("CAL8330 - Benjamin Pope")
program_path = os.path.join(data_dir, "8330/calslope/")
files = get_files(program_path, "nis_calslope")
calbin_files += files
calbin_files = [file for file in calbin_files if file[0].header["NGROUPS"] > 3]
calbin_files = [add_badpix(file) for file in calbin_files]
calbin_files = sort_files(calbin_files, n_files_per_filt)
summarise_files(calbin_files)


CAL8330 - Benjamin Pope
  program    target filter dither     g/i        date              PI    CAL
0    8330  V-EZ-Aqr  F480M   9/10  18/220  11-11-2025  Pope, Benjamin  False
1    8330  V-EZ-Aqr  F480M  10/10  18/220  11-11-2025  Pope, Benjamin  False


Flats.

# Exposures

In [6]:
from amigo.model_fits import PointFit, FlatFit
from retrain_fns import BinaryFit, DarkFit
from dorito.model_fits import ResolvedFit

# Construct the exposures
resolved_exposures = [ResolvedFit(file) for file in calbin_files]
# calbin_exposures = [BinaryFit(file) for file in calbin_files[0:2]]
# cal_exposures = calbin_exposures

exposures = [*resolved_exposures]
# exposures = [*resolved_exposures, *cal_exposures]

Finally, let's batch the exposures for training.

# Building model

In [ ]:
# from amigo.core_models import AmigoModel
# from amigo.optical_models import AMIOptics
# from amigo.detector_models import LinearDetector
# from amigo.ramp_models import NonLinearRamp
# from amigo.read_models import ReadModel
import dorito

# Get the model
# model = AmigoModel(
#     exposures=exposures,
#     optics=AMIOptics(),
#     detector=LinearDetector(),
#     ramp_model=NonLinearRamp(),
#     read=ReadModel(),
#     state=load_dict(amigo_files_path + "/calibration.npy"),
# )

# building the model
source_size = 61  # pixels
model = dorito.models.ResolvedAmigoModel(
    exposures=exposures,
    optics=amigo.optical_models.AMIOptics(),
    detector=amigo.detector_models.LinearDetector(),
    ramp_model=amigo.ramp_models.NonLinearRamp(),
    read=amigo.read_models.ReadModel(),
    state=load_dict(amigo_files_path + "/calibration.npy"),
    param_initers={
        "distribution": np.ones((source_size, source_size)) / source_size**2
    },
)

Populate from state.

In [ ]:
temp_files_path = "/home/dgxuser/max/data/amigo_files/temp"
best_state = load_dict(temp_files_path + "/scratch_final_state.npy")

print()
for key, value in best_state.items():
    print(key)
    if key in [
        'positions',
        'fluxes',
        'aberrations',
        'log_dist',
    ]:
        # continue
        if key in model.params.keys():
            # model = model.set(key, value)
            default = model.get(key)
            model = model.set(key, default | value)
        else:
            model = model.set(key, value)

# for exp in calbin_exposures:
#     model = model.set(
#         exp.map_param("pas"),
#         np.array(66.31),
#     )
#     model = model.set(
#         exp.map_param("separations"),
#         np.array(0.1259),
#     )
#     model = model.set(
#         exp.map_param("contrasts"),
#         np.array(0.376),
#     )
spec_dic = model.spectra
spec_dic["V-EZ-Aqr_F380M"] = np.array([-0.07376706, -0.07606107])
spec_dic["V-EZ-Aqr_F430M"] = np.array([-0.14906244, -0.14530317])
spec_dic["V-EZ-Aqr_F480M"] = np.array([-0.12971859, -0.12976371])
model = model.set("spectra", spec_dic)

model = model.set("spectra", jax.tree.map(lambda x: np.mean(x), model.get("spectra")))
print(model.spectra)


## Checking initial fits
Just looking at one per filter. Starting with the calibrator exposures.

In [ ]:
from amigo.plotting import summarise_fit

for exp in exposures:
    exp.print_summary()
    summarise_fit(model, exp)

# Optimisation

## Setup
For the neural network training we want to use a custom learning rate warmup and a temperature decay, so we will have to modify a lot of the statistical functions of amigo to be able to handle this

In [ ]:
from retrain_fns import (
    loss_fn,
    args_fn,
    looper_fn,
    # grads_fn,
    aux_fn,
    ff_reg,
    nl_reg,
    sep_reg,
    pa_reg,
    temp_decay,
    cosine_warmup,
)


def norm_fn(model_params, args):
    params = model_params.params
    if "log_dist" in params.keys():
        for k, log_dist in params["log_dist"].items():
            distribution = 10**log_dist
            params["log_dist"][k] = np.log10(distribution / distribution.sum())

    if "spectra" in params.keys():
        spectra = jax.tree.map(
            lambda x: np.clip(x, a_min=-0.8, a_max=0.8), params["spectra"]
        )
        params["spectra"] = spectra

    return model_params.set("params", params), args


# REGULARISATION
reg_dict = {
    "PA": (1.0, sep_reg),
    "SEP": (1.0, pa_reg),
}

Instantiating trainer class and updating fishers.

In [ ]:
from amigo.calibration import ValBatchedTrainer, BatchedTrainer, Trainer  # , looper_fn, aux_fn
from dorito.stats import apply_regularisers
from retrain_fns import summarise_fn

# NOTE: This seems to nuke the jitter/SRF gradient??
linear_model = model.set(["psf_upsample", "bleed"], [1, False])

# Trainer class
trainer_class = Trainer

trainer_class_kwargs = {
    # "loss_fn": loss_fn,
    # "args_fn": args_fn,
    # "grad_fn": grads_fn,
    "norm_fn": norm_fn,
    # "summarise_fn": summarise_fn,
    # "intermediate_prints": intermediate_prints,
    # "save_path": save_path,
    "cache": cache_dir,
}

trainer = trainer_class(**trainer_class_kwargs)

# trainer = trainer.update_fishers(
#     linear_model,
#     calbin_exposures,
#     # exposures,
#     parameters=[
#         "pas",
#         "separations",
#         "contrasts",
#         "positions",
#         "fluxes",
#         "aberrations",
#         # "spectra",
#     ],
#     # recalculate=recalculate_flag,
#     # overwrite=recalculate_flag,
# )

trainer = trainer.update_fishers(
    linear_model,
    resolved_exposures,
    # exposures,
    parameters=[
        "positions",
        "fluxes",
        # "spectra",
    ],
    # recalculate=recalculate_flag,
    # overwrite=recalculate_flag,
)

Training.

In [ ]:
from amigo.fitting import sgd, adam

# Define the optimisers
optimisers = {
    "log_dist": adam(5e-3, 0),
    "fluxes": sgd(1e-3, 0),
    "positions": sgd(1e-3, 2),
    # "aberrations": sgd(1e1, 5),
    # "pas": sgd(2e-3, 5),
    # "separations": sgd(5e-4, 7),
    # "contrasts": sgd(5e-4, 8),
    # "spectra": sgd(1e-3, 80),
}

        
print(optimisers.keys())
# This seems to fix some recompile issues
def fn(x):
    if isinstance(x, jax.Array):
        if "i" in x.dtype.str:
            return x
        return np.array(x, dtype=float)
    return x
model = jtu.map(fn, model)


trainer_kwargs = {
    "model": model,
    "optimisers": optimisers,
    "epochs": epochs,
    "batches": exposures,
}

# with jax.disable_jit():
result = trainer.train(**trainer_kwargs)

Plotting the results of that fit!

# Saving results

In [ ]:
params_to_save = [
    "fluxes",
    "positions",
    "aberrations",
    "spectra",
    "log_dist",
]

final_params = {key: result.model.get(key) for key in params_to_save}

np.save(
    os.path.join(temp_files_path, "scratch_final_state.npy"),
    final_params,
    allow_pickle=True,
)

In [ ]:
################## PLOTTING LOSSES ###################

losses = list(result.losses.values())[0]
n_epoch = len(losses)
start = int(0.2 * n_epoch)
stop = -1
if stop < 0:
    stop = n_epoch + stop

amigo.plotting.plot_losses(losses, start=start)
    

################### PLOTTING HISTORY AND SUMMARISE FIT ###################
amigo.plotting.plot(result.history)

for exp in exposures:
    exp.print_summary()
    amigo.plotting.summarise_fit(result.model, exp)


In [ ]:
from dLux import utils as dlu

optics_diameter = 6.603464  # JWST aperture diameter in meters
wavel = 4.3e-6  # approximate F430M mean wavelength in meters

for idx, exp in enumerate(exposures):

    # only plot science exposures
    if exp.calibrator:
        continue

    dist = result.model.get_distribution(exp, rotate=False)
    fig, ax = plt.subplots(figsize=(6, 2.3))

    c0 = dorito.plotting.plot_result(
        ax,
        dist / dist.max(),
        pixel_scale=model.psf_pixel_scale / model.oversample,
        cmap="inferno",
        norm=mpl.colors.PowerNorm(0.6, vmin=0, vmax=1.0),
        diff_lim=0.5 * dlu.rad2arcsec(wavel / optics_diameter),
        # scale=1.,
    )
    fig.colorbar(c0)
    plt.show()